# Jour 3 : du texte OCR aux événements du calendrier

**Objectif de la journée** : transformer la sortie de l'OCR (le fichier `ocr_<alias>.json` produit au jour 2)
en **événements** au format du cours, puis les **publier** dans l'agenda partagé de votre squad.

```
ocr_champion.json  →  parsing (VOTRE travail)  →  événements  →  CalDAV (fourni)
```

Un événement suit le schéma du cours :

```python
{"title": "Équipe A", "date": "2026-03-12", "start_time": "14:00", "end_time": "18:00",
 "location": None, "confidence": 0.85}
```

(`start_time`, `end_time` et `location` peuvent valoir `None`.)

Champs **optionnels**, pour les plannings à codes (équipes, postes) : `"person"` (la personne concernée, lue
sur la ligne), `"all_day"` (`True` : journée entière, ex. un repos) et `"overnight"` (`True` : poste de nuit,
la fin tombe le lendemain).

**Règles de confiance** (chaque événement reçoit un score entre 0 et 1) :

| Confiance | Décision |
|---|---|
| ≥ 0,8 | **pré-coché** : proposé et accepté par défaut |
| 0,5 – 0,8 | **à vérifier** : proposé, mais l'utilisateur doit confirmer |
| < 0,5 | **non proposé** |

> Le **rappel** reste la métrique qui compte : un créneau raté est un créneau perdu. Mieux vaut proposer
> un événement douteux « à vérifier » que le laisser tomber en silence.

La publication CalDAV (`caldav_publier.py`) est **fournie** : vous écrivez le parsing.

## Organisation de la squad : une partie par membre

Le notebook est long pour une seule personne : **répartissez les fonctions**, chacun dans **sa copie** du
notebook (*Fichier → Enregistrer une copie*), puis le ou la lead colle les cellules terminées dans le notebook
commun. Un même `.ipynb` modifié à plusieurs (Colab partagé ou git) donne des conflits.

| Membre | Fonctions (section 3) | Tests de la section 4 |
|---|---|---|
| **Heures et texte** | `normaliser`, `extraire_heures`, `CODES` + `traduire_code` | normaliser, extraire_heures, traduire_code |
| **Dates** | `extraire_date`, `extraire_mois_annee` | extraire_date, bruit d'OCR |
| **Géométrie** | réglage `CLASSES_*`, `associer`, `associer_ligne` | associer, associer_ligne |
| **Lead : assemblage** | `confiance`, `parser`, seuils (section 6), publication (section 7) | confiance, puis tout le reste |

À trois : le lead prend aussi `CODES` et `traduire_code`.

Chaque fonction a ses exemples et son test : la section 4 affiche `A FAIRE` pour une fonction pas encore
écrite, sans bloquer les autres. Seule dépendance : `extraire_date` appelle `normaliser`. En attendant la
vraie version, le membre « Dates » remplace dans **sa copie** le `raise NotImplementedError` de `normaliser`
par `return texte`.

**Parcours conseillé**

1. **Avant midi** : heures, dates, `associer` dans le cas le plus simple de votre planning, `parser`, puis une
   publication en **simulation** (section 7). Objectif : au moins une photo de bout en bout.
2. **Après-midi** : plannings à codes (`traduire_code`, `associer_ligne`, date par la colonne), `confiance`
   affinée, cas difficiles relevés sur vos photos, publication réelle sans doublon.
3. **Bonus** : en-tête sur deux mois (section 3.3), API `POST /parse` (section 8).

## 1. Paramètres

In [5]:
OCR_JSON = "ocr_champion.json"       # fichier produit par test_ocr_champion.ipynb
CREDENTIALS = "identifiants-hugo.b.json"   # fichier téléchargé sur votre profil du portail
PUBLIER = False                      # True = vraie publication sur Radicale (après la simulation !)
ANNEE = 2026                         # année par défaut quand elle n'apparaît pas sur le planning

## 2. Chargement des résultats OCR

Le fichier a cette forme :

```json
{"modele": "flexistaff_team-rocket@champion", "version": "3", "classes": {"0": "..."},
 "images": [{"image": "p1.jpg", "largeur": 1200, "hauteur": 800,
             "zones": [{"classe": "event", "confiance_detection": 0.91,
                        "x1": 10, "y1": 20, "x2": 110, "y2": 60,
                        "texte": "14h-18h Équipe A", "confiance_ocr": 0.87, "lignes_ignorees": 0}]}]}
```

In [6]:
import json
from collections import Counter
from pathlib import Path

with open(OCR_JSON, encoding="utf-8") as f:
    ocr = json.load(f)

print("Modèle :", ocr["modele"], "| version", ocr["version"])
print("Classes :", ocr["classes"])
for img in ocr["images"]:
    compte = Counter(z["classe"] for z in img["zones"])
    print(f"{img['image']} ({img['largeur']}x{img['hauteur']}) : {dict(compte)}")

Modèle : flexistaff_team-magma@champion | version 10
Classes : {'0': 'Date', '1': 'Cellule', '2': 'Nom', '3': 'Titre', '4': 'Cellule_code'}
lot-test_p000_IMG_2908.jpg (1600x1200) : {'Cellule': 32, 'Nom': 9, 'Date': 6, 'Titre': 2}
lot-test_p001_IMG_2909.jpg (1600x1200) : {'Cellule': 39, 'Nom': 8, 'Date': 7, 'Titre': 2}
lot-test_p002_IMG_2910.jpg (1600x1200) : {'Cellule': 30, 'Titre': 1}
lot-test_p003_IMG_2911.jpg (1600x1200) : {'Cellule': 31, 'Titre': 1}
lot-test_p004_IMG_2912.jpg (1600x1200) : {'Cellule': 12, 'Date': 5, 'Nom': 4, 'Titre': 1}
lot-test_p005_IMG_2913.jpg (1600x1200) : {'Cellule': 31, 'Titre': 1}
lot-test_p006_IMG_2915.jpg (1600x1200) : {'Nom': 7, 'Cellule': 26, 'Date': 6, 'Titre': 1}
lot-test_p007_IMG_2916.jpg (1600x1200) : {'Cellule_code': 162, 'Date': 20, 'Nom': 12}
lot-test_p008_IMG_2917.jpg (1600x1200) : {'Cellule': 30, 'Titre': 1}
lot-test_p009_IMG_2918.jpg (1600x1200) : {'Nom': 11, 'Cellule_code': 331, 'Date': 32}


### Un coup d'œil aux textes lus, classe par classe

**Vos classes sont les vôtres** (pas forcément `date_cell` / `event`…). Repérez ici laquelle porte la date,
laquelle porte l'heure, laquelle porte le titre, et adaptez les constantes de la section suivante.

In [7]:
for img in ocr["images"][:2]:  # les 2 premières images suffisent pour se faire une idée
    print("=====", img["image"])
    for z in sorted(img["zones"], key=lambda z: (z["y1"], z["x1"])):
        print(f"{z['classe']:<16} ({z['x1']:>4},{z['y1']:>4})-({z['x2']:>4},{z['y2']:>4})  {z['texte']!r}")

===== lot-test_p000_IMG_2908.jpg
Titre            ( 210, 215)-(1375, 285)  'PLANNING SEMAINE 43'
Date             (1241, 302)-(1411, 327)  '24/10'
Date             ( 901, 303)-(1072, 329)  '22/10'
Date             (1072, 303)-(1241, 327)  '23/10'
Date             ( 730, 304)-( 901, 329)  '21/10'
Date             ( 558, 305)-( 731, 330)  '20/10'
Date             ( 386, 306)-( 558, 330)  '19/10'
Cellule          (1071, 325)-(1243, 395)  'Entrepot Nord'
Cellule          ( 901, 326)-(1074, 397)  'Qual 12'
Nom              ( 216, 328)-( 386, 398)  'Mme Fontaine'
Cellule          ( 385, 328)-( 560, 399)  '8h30à 17h30 Entepôt Nord'
Cellule          (1243, 391)-(1417, 462)  '9h à 18h Hangar €'
Cellule          ( 902, 393)-(1075, 465)  '9h à 18h site Salnt-Priest'
Cellule          ( 384, 395)-( 559, 468)  '1h à 22h Depor Corbas'
Cellule          ( 558, 395)-( 733, 468)  '8h3o à 17h30 Quel B'
Nom              ( 214, 396)-( 385, 467)  'M.Chevaller'
Cellule          (1247, 459)-(1419, 530)  '13hà 

## 3. Votre travail : les fonctions de parsing

Complétez les fonctions ci-dessous (chaque membre les siennes, voir « Organisation de la squad »), en vérifiant
à chaque fois leurs exemples.
Les exemples (`>>>`) montrent le résultat attendu ; la section 4 contient des tests automatiques.

Commencez par l'adaptation à vos classes : indiquez quelle classe joue quel rôle.

In [8]:
# TODO : adaptez ces noms de classes à VOS labels (voir le résumé ci-dessus)
CLASSES_EVENEMENT = {"Cellule, Cellule_code"}       # zones qui décrivent un créneau (heures, titre)
CLASSES_DATE = {"Date"}        # zones qui portent un jour (numéro, ou date complète)
CLASSES_ENTETE_MOIS = {"Titre"}  # zones qui portent le mois (et parfois l'année)
#CLASSES_ENTETE_JOUR = {"day_header_row"}  # en-têtes de colonnes (Lun, Mar...) ; peut ne pas exister chez vous
CLASSES_PERSONNE = {"Nom"}  # optionnel : planning à codes, zones qui portent le nom de la personne (ex. {"agent"}).
                          # Si CLASS_MAP les envoie vers le rôle `resource_header`, c'est cette ligne-là.

### 3.1 `normaliser(texte)` : corriger les confusions de l'OCR

L'OCR confond souvent lettres et chiffres. On corrige **seulement dans un contexte numérique**
(un `O` au milieu de « Octobre » ne doit pas devenir `0`).

Règles de départ : `O` → `0`, `I` ou `l` → `1`, `Z` → `2`, et espaces parasites autour de `/`, `:`, `h`, `-`.

In [9]:
import re

CONFUSIONS = {"O": "0", "o": "0", "I": "1", "l": "1", "Z": "2", "B": "8", "S": "5"}

def normaliser(texte: str) -> str:
    """Corrige les confusions lettre/chiffre de l'OCR, uniquement en contexte numérique.

    >>> normaliser("1Z/O3")
    '12/03'
    >>> normaliser("I4h-18h")
    '14h-18h'
    >>> normaliser("Octobre")      # pas de chiffre autour : on ne touche à rien
    'Octobre'
    >>> normaliser("14 h 30")
    '14h30'
    """
    # TODO : ex. remplacer O/I/l/Z seulement s'ils sont collés à un chiffre ou à un séparateur de date/heure,
    # puis resserrer les espaces autour de « h », « : », « / » et « - ».
    # Provisoire, pour les autres membres en attendant la vraie version : remplacez la ligne ci-dessous
    # par `return texte` (le texte inchangé).
    # raise NotImplementedError
    
    # 1. Remplacer O/I/l/Z par un chiffre quand ils sont à côté d'un chiffre.
    #    On répète tant que quelque chose change (pour « 1OO » par exemple).
    # 1. Cas particuliers autour du « h » : « 17hoo » -> « 17h00 », « Bhà » -> « 8hà »
    texte = re.sub(r"(\d)h[oO]{2}", r"\1h00", texte)
    texte = re.sub(r"(?<![A-Za-z])([BS])h(?=[\sàa\d-])", lambda m: CONFUSIONS[m.group(1)] + "h", texte)

    # 2. Remplacer O/I/l/Z/B/S par un chiffre quand ils sont à côté d'un chiffre.
    #    On répète tant que quelque chose change (pour « 1OO » par exemple).
    changement = True
    while changement:
        changement = False
        for i in range(len(texte)):
            if texte[i] in CONFUSIONS:
                avant = texte[i - 1] if i > 0 else ""
                apres = texte[i + 1] if i + 1 < len(texte) else ""
                if avant.isdigit() or apres.isdigit():
                    texte = texte[:i] + CONFUSIONS[texte[i]] + texte[i + 1:]
                    changement = True

    # 3. Resserrer les espaces autour de « h » et des séparateurs : / - :
    texte = re.sub(r"(\d) *h *(\d)", r"\1h\2", texte)            # 14 h 30 -> 14h30
    texte = re.sub(r"(\d) +h\b", r"\1h", texte)                  # 18 h    -> 18h
    texte = re.sub(r"([\dh]) *([/:-]) *(\d)", r"\1\2\3", texte)  # 12 / 03 -> 12/03
    return texte


### 3.2 `extraire_heures(texte)` : début et fin

Formats à couvrir : `14h-18h`, `14:00-18:00`, `de 14h à 18h`, `9h30`, `8h30 - 12h`…
Retour : `(start_time, end_time)` au format `"HH:MM"`, `None` si absent. Une heure seule donne `(début, None)`.

In [10]:
def extraire_heures(texte: str):
    """Renvoie (start_time, end_time) au format "HH:MM" ; None pour ce qui est absent.

    >>> extraire_heures("14h-18h")
    ('14:00', '18:00')
    >>> extraire_heures("de 14h à 18h Réunion")
    ('14:00', '18:00')
    >>> extraire_heures("14:00-18:00")
    ('14:00', '18:00')
    >>> extraire_heures("9h30")
    ('09:30', None)
    >>> extraire_heures("Congé")
    (None, None)
    """
    # TODO : une expression régulière (re) qui capture 1 ou 2 heures, avec minutes optionnelles.
    # Pensez à valider : 0 <= heure <= 23 et 0 <= minutes <= 59 (un « 25h » est une erreur d'OCR).
    # raise NotImplementedError
    
    texte = normaliser(texte)
    texte = re.sub(r"(?<=\d)[.,;](?=\d)", ":", texte)                  # 22.00 -> 22:00
    texte = re.sub(r"(?<!\d)(\d{2})(\d{2})(?!\d)", r"\1:\2", texte)   # 0900  -> 09:00

    heures = []
    # une heure = 1 ou 2 chiffres, puis « h » ou « : », puis des minutes facultatives
    for h, m in re.findall(r"(\d{1,2})[h:](\d{2})?", texte)[:2]:  # on garde les 2 premières
        h = int(h)
        m = int(m) if m else 0
        if 0 <= h <= 23 and 0 <= m <= 59:
            heures.append(f"{h:02d}:{m:02d}")
        else:  # valeur absurde (erreur d'OCR) : on garde la place avec None
            heures.append(None)

    debut = heures[0] if len(heures) > 0 else None
    fin = heures[1] if len(heures) > 1 else None
    return (debut, fin)
    

### 3.3 `extraire_date(texte, annee=ANNEE, mois=None)` : la date

Selon vos classes, la date peut venir :

- d'une **cellule de date** (`12`, `Mar 12`, `12/03`, `12/03/2026`) ;
- d'un **en-tête de colonne** (`Mar 12/03`) ;
- d'un **en-tête de mois** (`Mars 2026`) : il donne le mois (et l'année) quand la cellule n'a que le numéro du jour.

Retour : une date `"YYYY-MM-DD"` ou `None` si elle est introuvable ou impossible (31/02 → `None`).
Le paramètre `mois` (entier) sert quand le texte ne contient que le jour. **Adaptez cette fonction à votre planning.**

**Bonus : en-tête sur deux mois** (« Novembre 2027 / Décembre 2027 ») : les numéros de jour qui **repartent à 1**
après avoir atteint 28-31 appartiennent au mois **suivant** (et à l'année suivante après décembre).
Gardez donc le dernier numéro lu dans la ligne d'en-tête pour détecter le passage.

In [11]:
from datetime import date

MOIS = {"janvier": 1, "janv": 1, "fevrier": 2, "février": 2, "fevr": 2, "févr": 2,
        "mars": 3, "avril": 4, "avr": 4, "mai": 5, "juin": 6, "juillet": 7, "juil": 7,
        "aout": 8, "août": 8, "septembre": 9, "sept": 9, "octobre": 10, "oct": 10,
        "novembre": 11, "nov": 11, "decembre": 12, "décembre": 12, "dec": 12, "déc": 12}


def extraire_date(texte: str, annee: int = ANNEE, mois: int | None = None):
    """Renvoie une date "YYYY-MM-DD", ou None.

    >>> extraire_date("Mar . 1Z/O3")
    '2026-03-12'
    >>> extraire_date("12/03/2025")
    '2025-03-12'
    >>> extraire_date("12", mois=3)           # numéro du jour seul + mois connu par l'en-tête
    '2026-03-12'
    >>> extraire_date("12 mars")
    '2026-03-12'
    >>> extraire_date("31/02") is None        # date impossible
    True
    """
    # TODO : appelez normaliser(), puis essayez les formats du plus précis au moins précis.
    # Validez avec datetime.date(annee, mois, jour) : une ValueError => None.
    # raise NotImplementedError
    
    if annee is None:
        annee = date.today().year

    texte = normaliser(texte).lower()
    nombres = re.findall(r"\d+", texte)

    # Le mois en toutes lettres, s'il y en a un (« 12 mars », « 22 nov »)
    en_lettres = False
    for mot in re.findall(r"[a-zéû]+", texte):
        if mot in MOIS:
            mois = MOIS[mot]
            en_lettres = True

    # On déduit le jour (et le mois / l'année s'ils sont dans le texte)
    if en_lettres and nombres:                                      # 12 mars, 22 novembre 2027
        jour = int(nombres[0])
        if len(nombres) > 1 and len(nombres[-1]) == 4:
            annee = int(nombres[-1])
    elif nombres and len(nombres[-1]) >= 4 and "/" not in texte:    # M 2306 -> 23/06
        jour, mois = int(nombres[-1][:2]), int(nombres[-1][-2:])
    elif len(nombres) >= 3:                                         # 12/03/2025
        jour, mois, annee = int(nombres[0]), int(nombres[1]), int(nombres[2])
    elif len(nombres) == 2:                                         # 12/03
        jour, mois = int(nombres[0]), int(nombres[1])
    elif len(nombres) == 1 and mois is not None:                    # « 12 » + mois connu
        jour = int(nombres[0])
    else:
        return None

    if annee < 100:                                                 # 12/03/25 -> 2025
        annee += 2000

    # On vérifie que la date existe
    try:
        return date(annee, mois, jour).isoformat()
    except ValueError:
        return None


def extraire_mois_annee(texte: str, annee: int = ANNEE):
    """Lit un en-tête de mois (« Mars 2026 ») et renvoie (annee, mois), ou (annee, None).

    >>> extraire_mois_annee("Mars 2026")
    (2026, 3)

    Bonus : un en-tête sur deux mois (« Novembre 2027 / Décembre 2027 ») donne deux couples : voyez comment
    les distinguer (les jours qui repartent à 1 basculent sur le mois suivant).
    """
    # TODO : cherchez un nom de mois (MOIS) et une année à 4 chiffres.
    #raise NotImplementedError

### 3.4 `associer(zones)` : relier un événement à sa date

Une zone « événement » ne contient généralement pas sa date : il faut la **déduire de la géométrie**
(coordonnées `x1, y1, x2, y2`). Pistes, à choisir selon **votre** type de planning :

- **Calendrier mural** : la zone événement est *à l'intérieur* de la cellule de date (recouvrement des boîtes,
  ex. part de l'aire de l'événement contenue dans la cellule).
- **Planning en colonnes** : la date est l'en-tête de la **même colonne** (centres `x` alignés).
- **Planning en lignes** : la date est dans la cellule la plus proche **à gauche** sur la même ligne (centres `y` alignés).

Retour : pour chaque zone événement, la zone « date » associée (ou `None`), avec si possible un indicateur
de fiabilité (recouvrement faible = moins sûr).
- **Planning à codes** (équipe × jours) : le texte d'une cellule (`M06`, `N08`, `RH`...) ne contient **ni la
  personne ni la date**. La **personne** se trouve par la **ligne** (zone « agent » dont le centre vertical
  `y` est aligné avec celui de l'événement) ; la **date** par la **colonne** (en-tête de date dont le centre
  horizontal `x` est aligné), quand la cellule de date ne contient pas l'événement.

**Pistes selon les classes de votre squad** (vos boîtes ne suivent pas toutes le cas « l'événement est dans
sa cellule de date ») :

- **Team Rocket** : `Date_case` encadre le **texte** de la date, pas la case. En planning hebdo, c'est l'en-tête
  de la colonne ; en grille de mois, c'est le numéro du jour, en haut de la case. Dans les deux cas, la date
  d'un `Evenement` est la `Date_case` **de la même colonne** (recouvrement horizontal) la plus proche
  **au-dessus** de lui.
- **Team Magma** : en planning hebdo, `Date` est la case de la date (en-tête de colonne). En grille de mois, il
  n'y a pas de `Date` : la `Cellule` est la case du jour entière, et **son texte commence par le numéro du jour**
  (« 12 09:00 Réunion 14:00 Visite »). Lisez ce numéro, puis découpez le reste en un événement par heure
  trouvée. Une `Cellule` qui ne contient que son numéro est un jour vide : pas d'événement.
- **Team Aqua** : `ref_semaine_planning` contient **toute la rangée** des jours (« LUNDI 30/12 MARDI 31/12… »),
  parfois avec le titre. Le JSON ne donne que le texte de la boîte, pas la position de chaque date : lisez
  les dates dans l'ordre, puis partagez la **largeur** de la boîte en autant de colonnes égales (la **hauteur**
  si la rangée est verticale, planning transposé). Une `horaire_lieu_jour_tache` prend la date de la colonne
  où tombe son centre. Une case sans texte est un jour vide : pas d'événement. En grille de mois, faites comme
  Magma (numéro du jour en tête du texte).

In [12]:
def associer_ligne(zone_evt: dict, zones: list[dict]):
    """Renvoie la zone en tête de la même ligne que l'événement (à gauche, centres verticaux alignés), ou None.
 
    Générique : `zones` peut être la liste des personnes (planning classique : le nom est au début de la ligne)
    ou celle des dates (planning transposé : la date est au début de la ligne).
 
    >>> evt = {"x1": 120, "y1": 40, "x2": 180, "y2": 60, "texte": "M06"}
    >>> p1 = {"classe": "agent", "x1": 0, "y1": 38, "x2": 100, "y2": 62, "texte": "Roux Inès"}
    >>> p2 = {"classe": "agent", "x1": 0, "y1": 64, "x2": 100, "y2": 88, "texte": "Le Gall Tom"}
    >>> associer_ligne(evt, [p1, p2])["texte"]
    'Roux Inès'
    >>> d1 = {"classe": "Date", "x1": 0, "y1": 38, "x2": 100, "y2": 62, "texte": "23 mars"}
    >>> associer_ligne(evt, [d1])["texte"]          # même fonction, avec des dates
    '23 mars'
    """
    # Le milieu vertical de l'événement : à mi-chemin entre son haut (y1) et son bas (y2)
    milieu_evt = (zone_evt["y1"] + zone_evt["y2"]) / 2
 
    # La zone d'en-tête couvre toute la hauteur de sa ligne : l'événement est sur cette ligne si son milieu
    # tombe entre le haut et le bas de la zone. Elle doit aussi être À GAUCHE de l'événement : c'est le début
    # de la ligne. Ce que contient la zone (un nom, une date) ne compte pas ici.
    for zone in zones:
        meme_ligne = zone["y1"] <= milieu_evt <= zone["y2"]
        a_gauche = zone["x2"] <= zone_evt["x1"] + 10  # 10 px de tolérance : les boîtes se touchent
        if meme_ligne and a_gauche:
            return zone
 
    return None  # aucune zone en tête de cette ligne : pas de réponse fiable
 
 
def associer_colonne(zone_evt: dict, zones: list[dict]):
    """Renvoie la zone en tête de la même colonne que l'événement (au-dessus, centres horizontaux alignés), ou None.
 
    Générique : `zones` peut être la liste des dates (planning classique : la date est en haut de la colonne)
    ou celle des personnes (planning transposé : le nom est en haut de la colonne).
 
    >>> evt = {"x1": 120, "y1": 40, "x2": 180, "y2": 60, "texte": "M06"}
    >>> d1 = {"classe": "Date", "x1": 110, "y1": 0, "x2": 190, "y2": 30, "texte": "23/03"}
    >>> d2 = {"classe": "Date", "x1": 190, "y1": 0, "x2": 270, "y2": 30, "texte": "24/03"}
    >>> associer_colonne(evt, [d1, d2])["texte"]
        '23/03'
    >>> p1 = {"classe": "Nom", "x1": 110, "y1": 0, "x2": 190, "y2": 30, "texte": "Roux Inès"}
    >>> associer_colonne(evt, [p1])["texte"]        # même fonction, avec des personnes
        'Roux Inès'
    """
    # Le milieu horizontal de l'événement : à mi-chemin entre son bord gauche (x1) et son bord droit (x2)
    milieu_evt = (zone_evt["x1"] + zone_evt["x2"]) / 2
 
    # La zone d'en-tête couvre toute la largeur de sa colonne : l'événement est dans cette colonne si son milieu
    # tombe entre le bord gauche et le bord droit de la zone. Elle doit aussi être AU-DESSUS de l'événement :
    # c'est le haut de la colonne. Ce que contient la zone (un nom, une date) ne compte pas ici.
    for zone in zones:
        meme_colonne = zone["x1"] <= milieu_evt <= zone["x2"]
        au_dessus = zone["y2"] <= zone_evt["y1"] + 10  # 10 px de tolérance : les boîtes se touchent
        if meme_colonne and au_dessus:
            return zone
 
    return None  # aucune zone en tête de cette colonne : pas de réponse fiable

### 3.4 bis Plannings à codes : `traduire_code(texte)`

Dans un planning d'équipe, la cellule ne contient pas des heures mais un **code** (`M06`, `N08`, `RH`...).
La **table des codes** est fournie avec le planning (légende) : à vous de la recopier dans `CODES`.

| Catégorie | Exemple | Résultat |
|---|---|---|
| travail | `M06` (06:00-14:00) | créneau horaire |
| nuit | `N08` (20:00-08:00) | `overnight=True` : la fin tombe le lendemain |
| repos / congé | `RH` | `all_day=True`, sans heures |

> **RGPD** : un arrêt maladie (`MAL`) est une **donnée de santé**. Dans un agenda partagé, son titre doit
> être « **Absence** », jamais « Arrêt maladie ».

## traitement

In [13]:
# TODO : complétez avec TOUS les codes de la légende de VOTRE planning
CODES = {
    "G03": {"titre": "Garde 12h",              "debut": "07:00", "fin": "19:00"},
    "G05": {"titre": "Garde 10h",              "debut": "08:00", "fin": "18:00"},
    "M06": {"titre": "Matin",                  "debut": "06:00", "fin": "14:00"},
    "S14": {"titre": "Soir",                   "debut": "14:00", "fin": "22:00"},
    "J09": {"titre": "Journée",                "debut": "09:00", "fin": "17:00"},
    "N08": {"titre": "Nuit",                   "debut": "20:00", "fin": "08:00"},
    "F0R": {"titre": "Formation",              "debut": "09:00", "fin": "17:00"},
    "RH":  {"titre": "Repos hebdomadaire",     "debut": None,    "fin": None},
    "RC":  {"titre": "Repos compensateur",     "debut": None,    "fin": None},
    "RF":  {"titre": "Repos férié",            "debut": None,    "fin": None},
    "CA":  {"titre": "Congé annuel",           "debut": None,    "fin": None},
    "MAL": {"titre": "Absence",                "debut": None,    "fin": None},
}

# ocr duper : 
# # 1re position = lettre attendue : un chiffre lu là est une lettre déformée
VERS_LETTRE = {"5": "S", "6": "G", "8": "B", "0": "O", "C": "G"}
# positions suivantes = chiffres attendus : une lettre lue là est un chiffre déformé
VERS_CHIFFRE = {"O": "0", "Q": "0", "D": "0", "S": "5", "B": "8", "G": "6",
                "I": "1", "Z": "2"}

def _corriger(code: str) -> str:
    """Corrige position par position : lettre en tête, chiffres ensuite."""
    if len(code) != 3:
        return code
    tete = VERS_LETTRE.get(code[0], code[0])
    reste = "".join(VERS_CHIFFRE.get(c, c) for c in code[1:])
    return tete + reste

FAMILLES = {}
for _c in CODES:
    if len(_c) == 3 and _c[1].isdigit():
        FAMILLES.setdefault(_c[0], []).append(_c)

def _par_famille(code: str):
    """Code tronqué ou abîmé : si la 1re lettre ne désigne qu'UN code chiffré, c'est lui."""
    if len(code) < 2:
        return None
    tete = VERS_LETTRE.get(code[0], code[0])
    reste = "".join(VERS_CHIFFRE.get(c, c) for c in code[1:])
    if not reste[0].isdigit():               # "MA..." = MAL, pas M06 : on ne touche pas
        return None
    famille = FAMILLES.get(tete, [])
    return famille[0] if len(famille) == 1 else None   # G03/G05 : ambigu -> None

# Règles de résolution des codes OCR
# REGLES = ("exact", "correction OCR", "parasite retiré", "GA -> CA", "famille unique", "non reconnu", "vide")

def resoudre_code(texte: str):
    """Renvoie (code_officiel, règle_utilisée), ou (None, "non reconnu")."""
    if not texte:
        return None, "vide"
    # Nettoyage : majuscules, on ne garde que lettres et chiffres ("J09 =" -> "J09")
    code = re.sub(r"[^A-Z0-9]", "", texte.upper())

    # Candidats, du plus sûr au plus corrigé ; le 1er qui existe gagne
    etapes = [(code, "exact"), (_corriger(code), "correction OCR")]
    if len(code) == 4:                                   # "INO8" -> N08
        etapes.append((_corriger(code[1:]), "parasite retiré"))
    if code == "GA":                                     # C lu G -> CA
        etapes.append(("CA", "GA -> CA"))
    etapes.append((_par_famille(code), "famille unique"))  # "Mos" -> M06

    for c, regle in etapes:
        if c in CODES:
            return c, regle
    return None, "non reconnu"


def traduire_code(texte: str):
    """Traduit un code de planning en (titre, start_time, end_time, all_day, overnight), ou None si inconnu.

    >>> traduire_code("M06")
    ('Matin', '06:00', '14:00', False, False)
    >>> traduire_code("MO6")                  # bruit d'OCR : le O est un 0
    ('Matin', '06:00', '14:00', False, False)
    >>> traduire_code("N08")
    ('Nuit', '20:00', '08:00', False, True)
    >>> traduire_code("RH")                   # repos : journée entière
    ('Repos hebdomadaire', None, None, True, False)
    >>> traduire_code("MAL")[0]               # donnée de santé : jamais dans l'agenda
    'Absence'
    >>> traduire_code("???") is None
    True
    >>> traduire_code("514")[0], traduire_code("NOB")[0], traduire_code("Gos")[0]
    ('Soir', 'Nuit', 'Garde 10h')
    >>> traduire_code("Mos")[0], traduire_code("S1")[0], traduire_code("GA")[0]
    ('Matin', 'Soir', 'Congé annuel')
    >>> traduire_code("G6") is None, traduire_code("RA") is None, traduire_code("M") is None
    (True, True, True)
    """
    code, _ = resoudre_code(texte)
    if code is None:
        return None
    info = CODES[code]
    debut, fin = info["debut"], info["fin"]
    all_day = debut is None
    overnight = not all_day and fin <= debut
    return (info["titre"], debut, fin, all_day, overnight)

## Visualisation

In [14]:
#Ou c'est qu'il pète un plomb dans cellule_code ? Sert à identifier les codes OCR non reconnus.
from collections import Counter

inconnus, reconnus = Counter(), Counter()
for img in ocr["images"]:
    for z in img["zones"]:
        if z["classe"] != "Cellule_code":
            continue
        t = z.get("texte", "")
        if not t:
            continue
        (reconnus if traduire_code(t) else inconnus)[t] += 1

print("Reconnus :", dict(reconnus))
print("NON reconnus :", inconnus.most_common())

Reconnus : {'G03': 31, 'FOR': 4, 'S14': 19, '514': 14, 'RH': 52, 'GO5': 1, 'C03': 1, 'J09': 18, 'G05': 2, 'NoB': 3, 'GO3': 3, 'RC': 37, 'RF': 7, 'NOB': 6, 'CA': 26, 'Go5': 2, 'MO6': 8, 'Mo6': 2, 'No8': 2, 'Mos': 6, 'Rh': 20, 'Go3': 9, 'Rc': 1, 'Rf': 3, 'Gos': 9, 'Non': 1, 'Noa': 4, 'Jo9': 6, 'S1': 5, 'Jos': 1, '(Gos': 1, 'FoR': 2, 'S16': 1, 'Ca': 1, '[Mos': 1, 'No]': 1, 'Fo': 1, 'co3': 1}
NON reconnus : [('NA', 1), ('oa', 1), ('09', 1), ('os', 1), ('M', 1), ('Ma', 1), ('Su', 1), ('vo', 1), ('006', 1), ('MA', 1), ('Ros', 1), ('RM', 1), ('Al', 1)]


## Rapport 

In [15]:
from collections import Counter

modifs = Counter()
for img in ocr["images"]:
    for z in img["zones"]:
        if z["classe"] != "Cellule_code":
            continue
        t = z.get("texte", "")
        code, regle = resoudre_code(t)
        modifs[(t, code, regle)] += 1

ordre = ["correction OCR", "parasite retiré", "GA -> CA", "famille unique", "non reconnu", "vide"]

print(f"{'Texte OCR':<14}{'→ Code':<9}{'Règle':<18}{'Nb':>4}")
print("-" * 45)
for (t, code, regle), n in sorted(modifs.items(),
                                  key=lambda kv: (ordre.index(kv[0][2]) if kv[0][2] in ordre else -1, -kv[1])):
    if regle == "exact":
        continue                              # lu correctement : rien à montrer
    print(f"{t!r:<14}{str(code or '—'):<9}{regle:<18}{n:>4}")

print("-" * 45)
par_regle = Counter()
for (_, _, regle), n in modifs.items():
    par_regle[regle] += n
for regle in ["exact"] + ordre:
    if par_regle[regle]:
        print(f"{regle:<20}{par_regle[regle]:>4}")

 

Texte OCR     → Code   Règle               Nb
---------------------------------------------
'514'         S14      correction OCR      14
'Go3'         G03      correction OCR       9
'Gos'         G05      correction OCR       9
'MO6'         M06      correction OCR       8
'NOB'         N08      correction OCR       6
'Jo9'         J09      correction OCR       6
'FOR'         F0R      correction OCR       4
'NoB'         N08      correction OCR       3
'GO3'         G03      correction OCR       3
'Go5'         G05      correction OCR       2
'Mo6'         M06      correction OCR       2
'No8'         N08      correction OCR       2
'FoR'         F0R      correction OCR       2
'GO5'         G05      correction OCR       1
'C03'         G03      correction OCR       1
'(Gos'        G05      correction OCR       1
'co3'         G03      correction OCR       1
'Mos'         M06      famille unique       6
'S1'          S14      famille unique       5
'Noa'         N08      famille uni

### 3.5 `confiance(...)` : combien fait-on confiance à l'événement ?

On part de **1,0** et on **retire** des points à chaque correction ou doute. À vous de choisir les pénalités
et de les **justifier** dans votre rapport. Idées :

- caractère corrigé par `normaliser` (O→0…) : petite pénalité ;
- confiance OCR faible de la zone (`confiance_ocr`) ou de la détection (`confiance_detection`) ;
- aucune heure trouvée (événement « toute la journée » par défaut) ;
- association à la date peu sûre (faible recouvrement) ;
- heure de fin ≤ début (erreur d'OCR).

Le résultat est borné entre 0 et 1.

In [94]:
def confiance(zone_evt: dict, zone_date: dict | None, score_association: float,
              texte_brut: str, texte_normalise: str, heures: tuple) -> float:
    """Score de confiance de l'événement, entre 0 et 1.

    >>> 0.0 <= confiance({"confiance_ocr": 0.9, "confiance_detection": 0.9}, None, 0.0, "x", "x", (None, None)) <= 1.0
    True
    """

    MALUS_TEXTE_NORMALISE = 0.1
    MALUS_HEURES_ABSENTES = 0.1
    MALUS_HEURES_INCOHERENTES = 0.1
    MALUS_ASSOCIATION_INCERTAINE = 0.1
    MALUS_SCORE_OCRDETEC_FAIBLE = 0.1
    
    # Score de confiance de l'événement
    score = 1.0
    
    # Réduire score si texte normalisé différent du texte brut
    if texte_brut != texte_normalise:
        score -= MALUS_TEXTE_NORMALISE

    # Réduire score si aucune heure trouvée
    if heures is None or heures[0] is None or heures[1] is None:
        score -= MALUS_HEURES_ABSENTES
    else:
        # Réduire score si heure de fin inférieure ou égale à heure de début
        if heures[1] <= heures[0]:
            score -= MALUS_HEURES_INCOHERENTES

    # Réduire score si association à la date peu sûre
    if score_association <= 0.6:
        score -= MALUS_ASSOCIATION_INCERTAINE

    # Réduire score si score de confiance ocr/détection trop faible
    if zone_evt["confiance_ocr"] is None or zone_evt["confiance_detection"] is None:
        pass
    else:
        if zone_evt["confiance_ocr"] <= 0.6 or zone_evt["confiance_detection"] <= 0.6:
            score -= MALUS_SCORE_OCRDETEC_FAIBLE

    print(score)
    return max(0.0, min(1.0, score))

### 3.6 `parser(ocr)` : assembler le tout

`parser` parcourt les images, associe chaque événement à sa date, extrait titre et heures, calcule la confiance,
et renvoie la liste des événements **au schéma du cours**. Le titre est le texte de la zone sans les heures.

In [96]:
def parser(ocr: dict, annee: int = ANNEE) -> list[dict]:
    """Transforme le JSON de l'OCR en liste d'événements (schéma du cours).

    Chaque élément : {"title", "date", "start_time", "end_time", "location", "confidence"}
    (+ "person", "all_day", "overnight" pour un planning à codes).
    Un événement dont la date est introuvable n'est pas perdu : mettez date=None et une confiance basse
    (ou écartez-le de façon explicite : décrivez votre choix dans le rapport).
    """
    events = list()
    for image in ocr["images"]:

        associations = list()
        zones = image["zones"]
        zones_evt = [zone_evt for zone_evt in zones if zone_evt["classe"] == "Cellule"]
        zones_date = [zone_date for zone_date in zones if zone_date["classe"] == "Date"]
        zones_nom = [zone_nom for zone_nom in zones if zone_nom["classe"] == "Nom"]
        zones_evt_code = [zone_evt_code for zone_evt_code in zones if zone_evt_code["classe"] == "Cellule_code"]
        zones_titre = [zone_titre for zone_titre in zones if zone_titre["classe"] == "Titre"]

        #print("len zones_evt", len(zones_evt))
        #print("len zones_date", len(zones_date))
        #print("len zones_nom", len(zones_nom))
        #print("len zones_evt_code", len(zones_evt_code))
        #print("len zones_titre", len(zones_titre))

        for zone_evt in zones_evt:

            association = associer_ligne(zone_evt, zones_date)
            if association is None:
                association = associer_colonne(zone_evt, zones_date)
            associations.append((zone_evt, association, 1))

        for association in associations:

            evt = association[0]
            date = association[1]

            _title = normaliser(evt["texte"])
            _date = extraire_date(date["texte"]) if date is not None else "null"
            _heures = extraire_heures(_title)
            _start_time = _heures[0] if _heures[0] is not None else "null"
            _end_time = _heures[1] if _heures[1] is not None else "null"
            _location = "null"
            _confidence = confiance(evt, date, 1, evt["texte"], _title, _heures)

            event = {
                "title": _title,
                "date": _date,
                "start_time": _start_time,
                "end_time": _end_time,
                "location": _location,
                "confidence": _confidence,
            }

            events.append(event)

    return events

print(len(ocr["images"]))
parser(ocr, 2026)

10
1.0
0.9
1.0
0.9
0.9
1.0
1.0
0.9
1.0
0.9
1.0
1.0
1.0
0.9
1.0
0.9
1.0
1.0
0.9
0.9
1.0
1.0
0.9
1.0
0.9
1.0
1.0
0.9
0.9
1.0
1.0
0.9
1.0
1.0
0.9
0.9
1.0
0.9
1.0
1.0
1.0
1.0
1.0
0.9
0.9
1.0
0.9
0.9
0.8
1.0
0.9
0.9
1.0
0.8
1.0
0.8
1.0
0.9
1.0
1.0
0.9
1.0
1.0
1.0
1.0
1.0
0.9
1.0
1.0
1.0
0.9
0.9
0.9
0.8
0.9
0.9
0.9
0.9
0.9
1.0
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.8
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
1.0
0.9
0.9
0.9
0.8
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
1.0
0.9
1.0
1.0
0.9
0.9
0.7000000000000001
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
1.0
0.9
1.0
1.0
1.0
1.0
1.0
1.0
0.8
1.0
1.0
1.0
0.9
1.0
1.0
1.0
1.0
0.9
1.0
0.9
0.9
1.0
1.0
1.0
0.9
0.8
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9
0.9


[{'title': '9h à 18h Hangar €',
  'date': '2026-10-24',
  'start_time': '09:00',
  'end_time': '18:00',
  'location': 'null',
  'confidence': 1.0},
 {'title': 'Entrepot Nord',
  'date': '2026-10-23',
  'start_time': 'null',
  'end_time': 'null',
  'location': 'null',
  'confidence': 0.9},
 {'title': '8h30 à 17h30 Entrepot Nord',
  'date': '2026-10-21',
  'start_time': '08:30',
  'end_time': '17:30',
  'location': 'null',
  'confidence': 1.0},
 {'title': '6h0 14h Depot Corbas',
  'date': '2026-10-19',
  'start_time': '06:00',
  'end_time': '14:00',
  'location': 'null',
  'confidence': 0.9},
 {'title': '8h30 à 17h30 Quel B',
  'date': '2026-10-20',
  'start_time': '08:30',
  'end_time': '17:30',
  'location': 'null',
  'confidence': 0.9},
 {'title': '8hà 16h Chander Part-Dleu',
  'date': '2026-10-23',
  'start_time': '08:00',
  'end_time': '16:00',
  'location': 'null',
  'confidence': 1.0},
 {'title': '8h30à 17h30 Entepôt Nord',
  'date': '2026-10-19',
  'start_time': '08:30',
  'end_t

## 4. Tests : faites-les passer

Quelques cas écrits à la main, dont du **bruit d'OCR**. Quand une fonction n'est pas encore écrite,
le test correspondant échoue avec `NotImplementedError` : normal. Avancez fonction par fonction.

In [ ]:
def tester(nom, fonction):
    try:
        fonction()
        print(f"OK     {nom}")
    except NotImplementedError:
        print(f"A FAIRE {nom}")
    except AssertionError as e:
        print(f"ECHEC  {nom} : {e}")


def test_normaliser():
    assert normaliser("1Z/O3") == "12/03"
    assert normaliser("I4h-18h") == "14h-18h"
    assert normaliser("Octobre") == "Octobre"


def test_heures():
    assert extraire_heures("14h-18h") == ("14:00", "18:00")
    assert extraire_heures("14:00-18:00") == ("14:00", "18:00")
    assert extraire_heures("de 14h à 18h") == ("14:00", "18:00")
    assert extraire_heures("9h30") == ("09:30", None)
    assert extraire_heures("Congé") == (None, None)
    assert extraire_heures("I4h-18h") == ("14:00", "18:00")  # bruit d'OCR : pensez à normaliser d'abord


def test_dates():
    assert extraire_date("12/03/2025") == "2025-03-12"
    assert extraire_date("12", mois=3) == "2026-03-12"
    assert extraire_date("31/02") is None


def test_bruit_ocr():
    # Le cas de référence : date ET heures dans un texte bruité
    texte = "Mar . 1Z/O3  I4h-18h"
    assert extraire_date(texte) == "2026-03-12", extraire_date(texte)
    assert extraire_heures(texte) == ("14:00", "18:00"), extraire_heures(texte)


def test_association():
    evt = {"classe": "event", "x1": 12, "y1": 30, "x2": 90, "y2": 50, "texte": "14h-18h A"}
    j12 = {"classe": "date_cell", "x1": 0, "y1": 0, "x2": 100, "y2": 100, "texte": "12"}
    j13 = {"classe": "date_cell", "x1": 100, "y1": 0, "x2": 200, "y2": 100, "texte": "13"}
    assert associer([evt, j12, j13])[0][1]["texte"] == "12"


def test_confiance():
    z = {"confiance_ocr": 0.95, "confiance_detection": 0.9}
    propre = confiance(z, {}, 1.0, "14h-18h", "14h-18h", ("14:00", "18:00"))
    bruite = confiance(z, {}, 1.0, "I4h-18h", "14h-18h", ("14:00", "18:00"))
    assert 0 <= bruite < propre <= 1, (bruite, propre)  # une correction doit coûter de la confiance


def test_associer_ligne():
    evt = {"x1": 120, "y1": 40, "x2": 180, "y2": 60, "texte": "M06"}
    p1 = {"x1": 0, "y1": 38, "x2": 100, "y2": 62, "texte": "Roux Inès"}
    p2 = {"x1": 0, "y1": 64, "x2": 100, "y2": 88, "texte": "Le Gall Tom"}
    assert associer_ligne(evt, [p2, p1])["texte"] == "Roux Inès"
    assert associer_ligne(evt, []) is None


def test_codes():
    assert traduire_code("M06") == ("Matin", "06:00", "14:00", False, False)
    assert traduire_code("MO6") == ("Matin", "06:00", "14:00", False, False)  # bruit d'OCR
    titre, debut, fin, journee, nuit = traduire_code("N08")
    assert nuit and not journee and (debut, fin) == ("20:00", "08:00")
    assert traduire_code("RH")[3] is True  # repos : journée entière
    assert traduire_code("MAL")[0] == "Absence"  # RGPD : pas de donnée de santé
    assert traduire_code("???") is None


for nom, f in [("normaliser", test_normaliser), ("extraire_heures", test_heures),
               ("extraire_date", test_dates), ("bruit d'OCR", test_bruit_ocr),
               ("associer", test_association), ("confiance", test_confiance),
               ("associer_ligne", test_associer_ligne), ("traduire_code", test_codes)]:
    tester(nom, f)

## 5. Lancer le parsing sur vos résultats OCR

In [ ]:
evenements = parser(ocr)
print(len(evenements), "événement(s) extrait(s)")
if not evenements:
    print("ATTENTION : 0 événement. parser() n'est pas encore écrit (section 3.6), "
          "ou il ne reconnaît pas vos classes (section 3, CLASSES_...).")
for e in evenements[:5]:
    print(e)

## 6. Tableau de revue : trois bandes de confiance

- **≥ 0,8** : pré-coché ;
- **0,5 – 0,8** : à vérifier (relisez-les : le parsing a corrigé ou deviné quelque chose) ;
- **< 0,5** : non proposé.

Vous ne publiez que les événements pré-cochés **et** ceux que vous avez vérifiés.

In [ ]:
SEUIL_PRECOCHE = 0.8
SEUIL_PROPOSE = 0.5

precoches = [e for e in evenements if e["confidence"] >= SEUIL_PRECOCHE]
a_verifier = [e for e in evenements if SEUIL_PROPOSE <= e["confidence"] < SEUIL_PRECOCHE]
non_proposes = [e for e in evenements if e["confidence"] < SEUIL_PROPOSE]

try:
    import pandas as pd
    for titre, liste in [("Pré-cochés", precoches), ("À vérifier", a_verifier), ("Non proposés", non_proposes)]:
        print(f"\n== {titre} : {len(liste)}")
        if liste:
            display(pd.DataFrame(liste).sort_values(["date", "start_time"], na_position="last"))
except ImportError:  # pandas absent : affichage simple
    for titre, liste in [("Pré-cochés", precoches), ("À vérifier", a_verifier), ("Non proposés", non_proposes)]:
        print(f"\n== {titre} : {len(liste)}")
        for e in liste:
            print(" ", e)

## 7. Publication dans Radicale (fournie)

Le module `caldav_publier.py` envoie vos événements dans l'agenda **`calevent`** de votre squad
(l'agenda partagé ; votre agenda personnel n'est pas touché). Il garantit :

- des heures en `Europe/Paris` et un identifiant (UID) stable : renvoyer le même créneau **ne crée pas de doublon** ;
- un événement « toute la journée » quand il n'y a pas d'heure de début, une durée de 1 h sans heure de fin.

**Où trouver le module ?**

- en local : lancez ce notebook depuis le dossier `ressources/` (ou copiez `caldav_publier.py` à côté du notebook) ;
- dans Colab : déposez le fichier dans le panneau *Fichiers*, ou téléchargez-le avec la cellule ci-dessous.

In [ ]:
%pip install -q caldav icalendar

# Colab uniquement, si caldav_publier.py n'est pas à côté du notebook :
if not Path("caldav_publier.py").exists():
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/CassiaFa/flexistaff/main/ressources/caldav_publier.py",
        "caldav_publier.py")

### Simulation d'abord (aucune connexion)

`dry_run=True` construit les événements iCalendar et affiche le statut `simulation` : un bon moyen de repérer
une date absente ou une heure invalide (statut `erreur: ...`) avant d'écrire dans l'agenda de l'équipe.

In [ ]:
from caldav_publier import publier, charger_identifiants

a_publier = precoches + a_verifier  # TODO : ne gardez dans a_verifier que ce que vous avez relu
simulation = publier(a_publier, url="", utilisateur="", mot_de_passe="", dry_run=True)
for r in simulation:
    print(f"{r['statut']:<12} {r['date']}  {r['titre']}")

### Publication réelle

Passez `PUBLIER = True` dans les paramètres **seulement** quand la simulation est propre.
Les statuts possibles : `créé`, `déjà présent` (relancer la cellule est sans danger), `erreur: ...`.
Le fichier d'identifiants n'est jamais affiché ni écrit dans le code.

In [ ]:
if PUBLIER:
    ident = charger_identifiants(CREDENTIALS)          # url, utilisateur, mot_de_passe
    resultats = publier(a_publier, **ident)             # agenda "calevent" par défaut
    for r in resultats:
        print(f"{r['statut']:<14} {r['date']}  {r['titre']}")
    with open(CREDENTIALS, encoding="utf-8") as f:
        print("\nVérifiez le résultat dans l'interface web de Radicale :", json.load(f)["caldav"]["web_url"])
else:
    print("PUBLIER = False : rien n'a été envoyé. Passez-le à True après la simulation.")

## 8. Questions pour votre rapport du jour 3

Répondez dans vos livrables, **chiffres à l'appui** (sur les images du jeu de test) :

1. **Seuils** : quels seuils de confiance avez-vous retenus (0,8 / 0,5 ou d'autres) ? Pourquoi ?
   Combien d'événements dans chaque bande ? En avez-vous vu un mal classé (bon événement non proposé, ou faux événement pré-coché) ?
2. **Pénalités** : quelles pénalités avez-vous choisies dans `confiance` et pourquoi ces valeurs ?
3. **Rappel** : combien de créneaux réels manquent à la fin (zone non détectée, texte vide, date introuvable) ?
   À quelle étape se perdent-ils : détection, OCR ou parsing ?
4. **Ce qui échoue** : quels textes OCR votre parser ne sait pas lire ? Donnez 3 exemples concrets et une piste d'amélioration.
5. **Association** : quelle règle géométrique avez-vous utilisée pour relier événement et date ? Dans quel cas se trompe-t-elle ?
6. **Détection et parsing** : le challenger aurait-il changé le résultat ? Qu'auriez-vous demandé au modèle de mieux détecter ?
7. **Bonus (optionnel)** : exposez `parser` dans une petite API FastAPI `POST /parse` (voir le guide).